# 🕵️ Case file 01 — The Gold bag leak

> **Mission briefing.** It is 09:14 at Skyline Travel and the support queue is on fire. Customers are being quoted baggage rules that are not theirs. You are the Observability Detective on duty. The answers sound confident, so the only way to find out what went wrong is to open the activity logs.

Before answering, the assistant searches a policy database and passes what it finds along with the question. In technical terms this is retrieval-augmented generation (RAG). Every run leaves an **activity log** (Langfuse calls it a *trace*). It is made of **steps** (*spans*): the search step and the answer step (*generation*).

## Case objective
Work out which policies the assistant was allowed to see, and how that changes the answer, its cost and its speed.

## What to expect
Three levels. Each run prints the answer and a link to its activity log. The evidence you need is in the activity log, not in the notebook.

## Bad practice
Trusting a confident answer without checking which policies it was based on.

## Good practice
Check the search step: what was searched, with which limits, and what came back.

## Why the good practice works
The assistant can only be as right as the policies it was given. The search step shows exactly what it was given.

## Deliverable
A claim code for each level you solve, and for your demo, one before/after pair of activity-log links.

## 🎮 How the game works

- Put your team name in the next cell and run it. Use **the same name in every notebook**.
- Each level asks you to find something in the activity logs. Type your answer in `check_flag(...)` and run it. Wrong guesses cost nothing.
- Correct answers give a **claim code**. Paste it into the game master's form.
- Stuck? `hint("1-1")` shows a hint. Each hint costs 25% of that level's points (two hints at most).
- Points: Level 1 = 100, Level 2 = 200, Level 3 = 300.
- Rumour has it a detective left a note somewhere in this case's activity logs. If you find it, submit it as level `1-B`.

In [1]:
from experiment_utils import check_flag, close_case, hint, run_rag, set_team, show_rag_results

TEAM = "your-team-name"  # ← your team name, the same in every notebook
set_team(TEAM)

⚠️  TEAM is still 'your-team-name'. Claim codes will not score: change it and rerun this cell.


'your-team-name'

## Level 1 — The wrong fare (100 points)

A **Basic**-fare customer asks how many checked bags they get. The assistant answers twice: once searching every policy, once searching only the customer's fare.

**Find:** the id of the policy the search returned when it searched every policy.

In [2]:
customer = "How many checked bags do I get on my booking?"
runs = [run_rag(customer, None, "no-filter"), run_rag(customer, "basic", "filter")]
show_rag_results(runs)


level 1 · all policies
  answer: If your booking is on the Gold fare, you get **one checked bag up to 23 kg**. A second checked bag costs **60 EUR**.
  tokens: 73 in / 123 out · latency: 1.9s
  activity log: http://localhost:3000/project/langfuse-observability/traces/808b11a330274dcbd2c60bf395d5a23b

level 1 · Basic fare only
  answer: On a Basic fare, **no checked bags are included**—only one cabin bag up to 7 kg. You can add a 23 kg checked bag for **45 EUR**.
  tokens: 83 in / 120 out · latency: 1.6s
  activity log: http://localhost:3000/project/langfuse-observability/traces/eb94d4fbf34d611de3fcab60e6341723


In [3]:
check_flag("1-1", "type your answer here")
# hint("1-1")

❌ Not quite. Try again, or run hint("1-1").


## Level 2 — Two questions, one answer (200 points)

A **Silver**-fare customer asks two things at once. The search is limited to their fare, so the policies are the right fare. Is the answer complete?

**Find:** the id of the policy the customer needed but the first search never handed to the assistant. Change `top_k` (how many policies the search returns) and rerun the second cell to compare.

In [4]:
question = "Can I bring my cat in the cabin, and what does a second checked bag cost?"
level2 = [run_rag(question, "silver", "top-1", level=2, query=question)]
show_rag_results(level2)


level 2 · Silver fare only
  answer: Yes—on a Silver fare you can bring one cat (or small dog) in the cabin for **50 EUR on selected routes**. The pet carrier counts as your cabin bag and must fit under the seat in front of you.

The cost of a **second checked bag is not specified** in the retrieved policy.
  tokens: 99 in / 136 out · latency: 2.3s
  activity log: http://localhost:3000/project/langfuse-observability/traces/0eccf1037f6b33619ca27a5689eda8e4


In [5]:
top_k = 1  # ← try a bigger number, then compare the two activity logs
level2.append(run_rag(question, "silver", f"top-{top_k}", level=2, query=question, top_k=top_k))
show_rag_results(level2[-1:])


level 2 · Silver fare only
  answer: Yes—on a Silver fare you can bring one cat (or small dog) in the cabin for **50 EUR on selected routes**. The pet carrier counts as your cabin bag and must fit under the seat in front of you.

The cost of a **second checked bag is not specified** in the retrieved policy.
  tokens: 99 in / 136 out · latency: 2.2s
  activity log: http://localhost:3000/project/langfuse-observability/traces/a04cda979ff001d30929fc319b2b5952


In [6]:
check_flag("1-2", "type your answer here")
# hint("1-2")

❌ Not quite. Try again, or run hint("1-2").


## Level 3 — Playing it safe gets expensive (300 points)

A colleague proposes a shortcut: *"Skip the search and send every policy to the assistant, to be safe."* The answer still looks fine. Compare the two runs: the words the assistant had to read (input tokens), the time it took, and the cost shown in Langfuse.

**Find:** how many policies the shortcut sent to the assistant.

In [7]:
level3 = [run_rag(customer, "basic", "filter", level=3),
          run_rag(customer, None, "send-all", level=3, send_all=True)]
show_rag_results(level3)


level 3 · Basic fare only
  answer: On a Basic fare, **no checked bags are included**—only one cabin bag up to 7 kg. You can add a 23 kg checked bag for **45 EUR**.
  tokens: 83 in / 120 out · latency: 1.6s
  activity log: http://localhost:3000/project/langfuse-observability/traces/f7e33f7333f7e7215654301fcf6755ec

level 3 · every policy sent
  answer: It depends on your fare type:

- **Basic:** No checked bag included. You get one cabin bag up to 7 kg; a 23 kg checked bag costs **45 EUR**.
- **Silver:** One 23 kg checked bag included. A second checked bag costs **60 EUR**.
- **Gold:** The retrieved policies conflict: one says **two 23 kg checked bags are included free**, while another says **one 23 kg checked bag is included and a second costs 60 EUR**. You may need to confirm your Gold booking’s allowance.

Skis, golf clubs, and bicycles travel as a checked item for **55 EUR** on every fare.
  tokens: 464 in / 689 out · latency: 9.7s
  activity log: http://localhost:3000/project/lan

In [8]:
check_flag("1-3", "type your answer here")
# hint("1-3")

❌ Not quite. Try again, or run hint("1-3").


In [9]:
# Found the detective's note?
check_flag("1-B", "type your answer here")

❌ Not quite. Try again, or run hint("1-B").


## 🏁 Wrap-up

Run the last cell to close the case and send the activity logs to Langfuse. Keep one before/after pair of links for your demo.

In [10]:
close_case("activity-1", runs + level2 + level3)


case activity-1 closed — session activity-1-7fea5bd7
level 1 · all policies: http://localhost:3000/project/langfuse-observability/traces/808b11a330274dcbd2c60bf395d5a23b
level 1 · Basic fare only: http://localhost:3000/project/langfuse-observability/traces/eb94d4fbf34d611de3fcab60e6341723
level 2 · Silver fare only: http://localhost:3000/project/langfuse-observability/traces/0eccf1037f6b33619ca27a5689eda8e4
level 2 · Silver fare only: http://localhost:3000/project/langfuse-observability/traces/a04cda979ff001d30929fc319b2b5952
level 3 · Basic fare only: http://localhost:3000/project/langfuse-observability/traces/f7e33f7333f7e7215654301fcf6755ec
level 3 · every policy sent: http://localhost:3000/project/langfuse-observability/traces/7a9049f809585a2c80b7d6ee420fee38
activity logs flushed to Langfuse
